## Staging
**Goal: clean raw data and prepare a standardized layer for the mart.**

- Fix data types 
- Standardize column names (consistent naming style)
- Remove technical noise
- Add helper fields (year, month) for convenience

In [ ]:
import duckdb
import pandas as pd
import os

pd.set_option('display.max_columns', None)

con = duckdb.connect()

RAW     = 'data/raw'
STAGING = 'data/staging'
os.makedirs(STAGING, exist_ok=True)

def q(sql):
    return con.sql(sql).df()

# Executes a query and saves the result as a CSV file in staging:
def save(sql, filename):  
    df = con.sql(sql).df()
    path = f'{STAGING}/{filename}.csv'
    df.to_csv(path, index=False)
    return df


### Table `instructors`

Standardization of data types.


In [ ]:
stg_instructors = save(f"""
SELECT
    instructor_id,
    TRIM(name)         AS instructor_name,   -- remove extra spaces
    LOWER(tier)        AS tier,              -- standardize case: top/mid/new
    TRIM(subject_area) AS subject_area
FROM '{RAW}/instructors.csv'
WHERE instructor_id IS NOT NULL
""", 'stg_instructors')

### Table `courses`
Added `price_tier` - a helper category.


In [ ]:
stg_courses = save(f"""
SELECT
    course_id,
    TRIM(title)                          AS title,
    TRIM(category)                       AS category,
    instructor_id,
    CAST(price AS DECIMAL(10,2))         AS price,
    CAST(duration_hours AS DECIMAL(6,1)) AS duration_hours,
    CASE
        WHEN CAST(price AS INTEGER) <=  9 THEN 'intro'     -- $9
        WHEN CAST(price AS INTEGER) <= 29 THEN 'standard'  -- $29
        WHEN CAST(price AS INTEGER) <= 49 THEN 'advanced'  -- $49
        ELSE                                   'premium'   -- $99
    END                                  AS price_tier

    

FROM '{RAW}/courses.csv'
WHERE course_id IS NOT NULL
  AND CAST(price AS FLOAT) > 0
  AND CAST(duration_hours AS FLOAT) > 0
""", 'stg_courses')



### Table `students` 
Added `cohort_month` - the student's registration month.

In [ ]:
stg_students = save(f"""
SELECT
    student_id,
    CAST(reg_date AS DATE)                      AS reg_date,
    UPPER(TRIM(country))                        AS country,
    TRIM(acquisition_channel)                   AS acquisition_channel,

    -- Cohort: registration month — the basis for retention and cohort analysis.
    DATE_TRUNC('month', CAST(reg_date AS DATE)) AS cohort_month,

    -- Registration year — used for year-over-year groupings.
    YEAR(CAST(reg_date AS DATE))                AS reg_year

FROM '{RAW}/students.csv'
WHERE student_id IS NOT NULL
  AND reg_date   IS NOT NULL
""", 'stg_students')

### Table `enrollments`
Added helper date fields for period-based aggregations in Tableau.

In [ ]:
stg_enrollments = save(f"""
SELECT
    enrollment_id,
    student_id,
    course_id,
    CAST(enroll_date    AS DATE)               AS enroll_date,
    

    -- Helper date fields for period-based aggregations in Tableau
    DATE_TRUNC('month', CAST(enroll_date AS DATE)) AS enroll_month,
    YEAR(CAST(enroll_date AS DATE))            AS enroll_year
    


FROM '{RAW}/enrollments.csv'
WHERE enrollment_id IS NOT NULL
  AND student_id    IS NOT NULL
  AND course_id     IS NOT NULL
""", 'stg_enrollments')

### Table `payments`
Added helper date fields, standardized positive amounts, created boolean flags, prepared net_amount for revenue analysis.

In [ ]:
stg_payments = save(f"""
SELECT
    payment_id,
    student_id,
    course_id,                                 

    -- Store the amount as always positive
    ABS(CAST(amount AS DECIMAL(10,2))) AS amount,

    LOWER(TRIM(status))                AS status,
    CAST(date AS DATE)                 AS payment_date,
    
    --Helper date fields for time-series aggregations 
    DATE_TRUNC('month', CAST(date AS DATE)) AS payment_month,
    YEAR(CAST(date AS DATE))           AS payment_year,

    -- Boolean flags 
    LOWER(status) = 'success'          AS is_success,
    LOWER(status) = 'refunded'         AS is_refunded,
    LOWER(status) = 'failed'           AS is_failed,

    -- Net amount - a key field for revenue metrics
    CASE
        WHEN LOWER(status) = 'success'
        THEN ABS(CAST(amount AS DECIMAL(10,2)))
        ELSE 0
    END                                AS net_amount

FROM '{RAW}/payments.csv'
WHERE payment_id IS NOT NULL
  AND student_id IS NOT NULL
  AND course_id  IS NOT NULL
  AND date       IS NOT NULL
""", 'stg_payments')